In [1]:
import os
from deepeval.synthesizer import Synthesizer
from deepeval.models import OpenRouterModel

In [2]:
deep_model = OpenRouterModel(
    "openai/gpt-oss-120b",
    api_key=os.getenv("NOVITA_API_KEY"),
    base_url="https://api.novita.ai/openai"
)

In [3]:
deep_model.generate("hi")

('Hello! How can I help you today?', None)

In [4]:
from typing import List, Optional
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from deepeval.models import DeepEvalBaseEmbeddingModel


class CustomEmbeddingModel(DeepEvalBaseEmbeddingModel):
    def __init__(self):
        pass

    def load_model(self):
        return OpenAIEmbeddings(
            model="qwen/qwen3-embedding-0.6b",
            api_key=os.getenv("NOVITA_API_KEY"),
            base_url="https://api.novita.ai/openai",
        )

    def embed_text(self, text: str) -> List[float]:
        embedding_model = self.load_model()
        return embedding_model.embed_query(text)

    def embed_texts(self, texts: List[str]) -> List[List[float]]:
        embedding_model = self.load_model()
        return embedding_model.embed_documents(texts)

    async def a_embed_text(self, text: str) -> List[float]:
        embedding_model = self.load_model()
        return await embedding_model.aembed_query(text)

    async def a_embed_texts(self, texts: List[str]) -> List[List[float]]:
        embedding_model = self.load_model()
        return await embedding_model.aembed_documents(texts)

    def get_model_name(self):
        "Custom Qwen3 Embedding Model"

In [5]:
custom_embedding_model = CustomEmbeddingModel()

In [6]:
from langchain_openai import ChatOpenAI
from deepeval.models.base_model import DeepEvalBaseLLM

class OpenSourceLLM(DeepEvalBaseLLM):
    def __init__(
        self,
        model
    ):
        self.model = model

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        return chat_model.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        chat_model = self.load_model()
        res = await chat_model.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return "Custom Azure OpenAI Model"

# Replace these with real values
custom_model = ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("NOVITA_API_KEY"),
    base_url="https://api.novita.ai/openai"
)
open_source_llm = OpenSourceLLM(model=custom_model)
print(open_source_llm.generate("Write me a joke"))

Why did the scarecrow become a stand‑up comedian?

Because he was outstanding in his field—and his jokes always *corn*-y!


In [7]:
files = [
    "/home/vasim/Downloads/Derivative Formula - What is Derivative Formula_ Examples.pdf",
    "/home/vasim/Downloads/UnderstandingDeepLearning_05_29_25_C.pdf",
    "/home/vasim/Downloads/The Ultimate Python Handbook.pdf",
]

In [8]:
# !uv add chromadb
# !uv add pypdf
# !uv add ipywidgets

In [9]:
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import ContextConstructionConfig


synthesizer = Synthesizer(model=open_source_llm)
synthesizer.generate_goldens_from_docs(
    context_construction_config=ContextConstructionConfig(
        embedder=custom_embedding_model,
        critic_model=open_source_llm
    ),
    document_paths=files
)

Output()

[Confident AI Synthesizer Log] SUCCESS: Successfully deleted: /tmp/deepeval_chroma_k09ws0v4

[Confident AI Synthesizer Log] SUCCESS: Context Construction: Utilizing 0 out of 0 chunks.

[]